In [1]:
import pandas as pd
import os 
from pathlib import Path
import numpy as np 
import pickle
import random
import re

In [2]:
def parse_response_text(response_text, to_string=True):
    
    clean_text = re.sub(r'[ \t\n\r\f\v\-_.,，、。！？!?:：;；]', '', response_text)

    # Total relevant characters
    total_chars = len(clean_text)

    # Count Chinese characters
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', clean_text))

    # Compute ratio
    chinese_ratio = chinese_chars / total_chars if total_chars > 0 else 0
    # print(f"Chinese ratio: {chinese_ratio:.2f} ({chinese_chars}/{total_chars})")
    # Set flag if ratio > 70%
    if chinese_ratio > 0.7:
        # print("Response text is likely in Chinese.")
        chinese_flag = True
    
    # if '，' in response_text or '、' in response_text:
        chinese_flag = True
        # remove anything before ：
        # print(f"response_text: {response_text}")
        response_text = re.sub(r'.*：', '', response_text)
        response_text = re.sub(',', '，', response_text)  # replace commas with Chinese commas
        response_text = re.sub('、', '，', response_text)  # replace Chinese commas with Chinese commas
        
        # if there is no '，', change ' ' to '，'
        if '，' not in response_text and response_text.count('\n') < 3:
            response_text = re.sub(' ', '，', response_text)
        
        # print(f"response_text after removing prefix: {response_text}")
    else:
        chinese_flag = False
    
    if response_text.isascii():
        # use , to split the response text
        response_words = [x.strip().lower() for x in response_text.split(',')]
    else:
        # use ， to split the response text
        response_words = [x.strip().lower() for x in response_text.split('，')]
    response_words_raw = response_words.copy()
        
 
    re_parsing_list = [
        r'[-\*]\s+(.+?)(?=\n-|\n*$)',
        r'^\d+\.\s+(.+)$',
    ]
    
    response_words_re = None 
    
    for pattern in re_parsing_list:
        temp_response_words_re = re.findall(pattern, response_text, flags=re.MULTILINE) # just handle the llama vanilla case 
        if response_words_re is None or len(temp_response_words_re) > len(response_words_re):
            response_words_re = temp_response_words_re
            
    if len(response_words_re) < 2:
        # print(f"doing additional regex parsing for response_text: {response_text}")
        response_words_re = re.findall(r'[A-Za-z0-9_-]+(?:[ \t]+[A-Za-z0-9_-]+){0,2}[\r\n]', response_text, re.MULTILINE)
    
    # print(f"response_words_re: {response_words_re}")
    # print(f"response_words: {response_words}")
    # print(len(response_words_re), len(response_words))
    if max(len(response_words_re), len(response_words)) < 2:
        response_words = "\n".join(response_words).split('\n')
        response_words = [word.strip().lower().strip('-') for word in response_words]
        
    elif len(response_words_re) > len(response_words):
        response_words = response_words_re
        response_words = [word.strip().lower().strip('-') for word in response_words]
        
    if response_text.count("\n") <= 3 and len(response_words_raw) < 4: 
        if not to_string:
            return [] 
        else:
            return ""
    
    
    
    # make it set to remove duplicates
    new_list = []
    for word in response_words:
        if chinese_flag:
            # print(f"word before parsing: {word}")
            parse_chinese = re.findall(r'[\u4e00-\u9fff]+', word)
            word = ''.join(parse_chinese)
        if word not in new_list and word != '':
            word = word.replace('  ', ' ')  # replace multiple spaces with a single space
            new_list.append(word)
        response_words = new_list
    if not to_string:
        
        return response_words
    else:
        return ", ".join(response_words)

In [3]:
with open('/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/notebooks/HLCP-A-eval-12-models/swow_us_results.pkl', 'rb') as f:
    swow_en_results = pickle.load(f)

with open('/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/notebooks/HLCP-A-eval-12-models/ppo_update_zh_oct_2025/swow_zh_results.pkl', 'rb') as f:
    swow_zh_results = pickle.load(f)

In [4]:
swow_en_results.columns

Index(['Cue Word', 'Ground Truth Associated Words', 'Prompt Type',
       'Model Type', 'Generated Associated Words'],
      dtype='object')

In [5]:
swow_en_results_gaw = swow_en_results['Generated Associated Words']
swow_zh_results_gaw = swow_zh_results['Generated Associated Words']

## Test English Parsing

In [6]:
random_ind = random.randint(0, len(swow_en_results_gaw) - 1)

response_text = swow_en_results_gaw[random_ind]

print("Generated Associated Words:")
print(response_text)

response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Generated Associated Words:
Given the cue word "timeline," here's a list of words I immediately associate with it:

1. History
2. Chronology
3. Schedule
4. Events
5. Calendar
6. Time
7. Progress
8. Map
9. Graph
10. Space

These words were generated based on my language patterns and associations learned from large text datasets. Keep in mind that personal experiences and memories can influence the strength of associations for a specific individual, but this is a general AI response. Do you have any questions or would you like to try another cue word?
Parsed Response List:
history, chronology, schedule, events, calendar, time, progress, map, graph, space


In [7]:
response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Parsed Response List:
history, chronology, schedule, events, calendar, time, progress, map, graph, space


## Test Chinese Parsing

In [6]:
random_ind = random.randint(0, len(swow_zh_results_gaw) - 1)

response_text = swow_zh_results_gaw[random_ind]

print("Generated Associated Words:")
print(response_text)

response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Generated Associated Words:
我会根据"公司"这个词试图联想出一些相关词语。下面是我的列表：

1. 公司的职员：职员、员工、工资
2. 公司的地点：办公室、总部、大楼
3. 公司的服务：产品、服务、供应商
4. 公司的部门：行销、研发、财务
5. 公司的关系：投资人、管理层、股东
6. 公司的管理：领导人、经理、大 boss

这些是立马想到的联想词，希望
Parsed Response List:
职员, 员工, 工资办公室, 总部, 大楼产品, 服务, 供应商行销, 研发, 财务投资人, 管理层, 股东领导人, 经理, 大这些是立马想到的联想词, 希望


In [7]:
response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Parsed Response List:
职员, 员工, 工资办公室, 总部, 大楼产品, 服务, 供应商行销, 研发, 财务投资人, 管理层, 股东领导人, 经理, 大这些是立马想到的联想词, 希望


In [9]:
# process swow_en_results_gaw
# swow_en_results['Parsed Associated Words'] = swow_en_results_gaw.apply(parse_response_text)

# # Save the processed results
# with open('/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/notebooks/HLCP-A-eval-12-models/swow_us_results_processed.pkl', 'wb') as f:
#     pickle.dump(swow_en_results, f)
    
    
# process swow_zh_results_gaw
swow_zh_results['Parsed Associated Words'] = swow_zh_results_gaw.apply(parse_response_text)

with open('./swow_zh_results_processed.pkl', 'wb') as f:
    pickle.dump(swow_zh_results, f)

In [10]:
swow_zh_results

,Cue Word,Ground Truth Associated Words,Prompt Type,Model Type,Generated Associated Words,Parsed Associated Words
0,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,sukai/qwen_ppo_zh,看守 手铐 警徽 笼头 警绳 执行 警察 白袍 防暴装备 监狱 逮捕 略ест 夺命 行刑 ...,"看守, 手铐, 警徽, 笼头, 警绳, 执行, 警察, 白袍, 防暴装备, 监狱, 逮捕, ..."
1,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,sukai/qwen_ppo_zh,好的，以下是与“狱警”有关的一些联想词列表：\n\n1. 犯人\n2. 狱室\n3. 监狱\...,"犯人, 狱室, 监狱, 安全, 约束, 正义, 执行, 纽扣, 制服, 监管"
2,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,sukai/llama_ppo_zh,狱警\n 囚犯\n 监狱\n 座机\n 勤奋\n 骑警\n 监守\n 保卫\njisuan\...,"jisuan, punishment"
3,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,sukai/llama_ppo_zh,"玩词联想实验 soundsIZES趣!给出的词是 ""狱警"",我的立即联想词是：\n\n1. ...","监狱, bornescort, 警察, 服役, 刑事, orderid真身, 犯罪, 救援"
4,香蕉,黄色， 水果， 苹果， 猴子， 猩猩， 甜， 好吃， 剥皮， 通便， banana， 小黄人...,Complex,sukai/qwen_ppo_zh,水果、黄色、甜味、隔壁树、可食用、弯曲、热带、营养、热带水果、蕉叶、光滑、夏季水果、热带植物...,"水果, 黄色, 甜味, 隔壁树, 可食用, 弯曲, 热带, 营养, 热带水果, 蕉叶, 光滑..."
...,...,...,...,...,...,...
4011,嗨,你好， 打招呼， 问好， 好， hello， hi， 开心， 哈喽， 微笑， 早上好， hi...,Simple,sukai/llama_ppo_zh,"好玩! 我们来玩这个词联想实验吧。给出的词是 ""嗨""。\n\n以下是快速联想到的词:\n\n...","问候, 高兴, 话, 打招呼, 运动, handling (专业用语：计量器, scient..."
4012,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Complex,sukai/qwen_ppo_zh,忏悔、赎罪、反省、悔悟、改过、自新、悔恨、悲痛、悔悟、补救、改正、归正、宽恕、检察、 يتو...,
4013,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Simple,sukai/qwen_ppo_zh,好的，对于词语“悔改”，以下是一些可能的联想词：\n\n1. 罪恶\n2. 反思\n3. 天...,"罪恶, 反思, 天堂, 原谅, 新开始, 错误, 到悔悟, 坦白, 忏悔, 失误"
4014,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Complex,sukai/llama_ppo_zh,悔改 \n深感遗憾\n后悔\n忏悔\n反思\n检讨\n悔恨\n感谢赦免...,"悔改, 深感遗憾, 后悔, 忏悔, 反思, 检讨, 悔恨, 感谢赦免, 忏�πτωση, 主..."


In [11]:
# remove rows where Parsed Associated Words has less than 2 ","
# swow_en_results = swow_en_results[swow_en_results['Parsed Associated Words'].str.count(",") >= 2]
swow_zh_results = swow_zh_results[swow_zh_results['Parsed Associated Words'].str.count(",") >= 2]

In [12]:
swow_zh_results

,Cue Word,Ground Truth Associated Words,Prompt Type,Model Type,Generated Associated Words,Parsed Associated Words
0,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,sukai/qwen_ppo_zh,看守 手铐 警徽 笼头 警绳 执行 警察 白袍 防暴装备 监狱 逮捕 略ест 夺命 行刑 ...,"看守, 手铐, 警徽, 笼头, 警绳, 执行, 警察, 白袍, 防暴装备, 监狱, 逮捕, ..."
1,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,sukai/qwen_ppo_zh,好的，以下是与“狱警”有关的一些联想词列表：\n\n1. 犯人\n2. 狱室\n3. 监狱\...,"犯人, 狱室, 监狱, 安全, 约束, 正义, 执行, 纽扣, 制服, 监管"
3,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,sukai/llama_ppo_zh,"玩词联想实验 soundsIZES趣!给出的词是 ""狱警"",我的立即联想词是：\n\n1. ...","监狱, bornescort, 警察, 服役, 刑事, orderid真身, 犯罪, 救援"
4,香蕉,黄色， 水果， 苹果， 猴子， 猩猩， 甜， 好吃， 剥皮， 通便， banana， 小黄人...,Complex,sukai/qwen_ppo_zh,水果、黄色、甜味、隔壁树、可食用、弯曲、热带、营养、热带水果、蕉叶、光滑、夏季水果、热带植物...,"水果, 黄色, 甜味, 隔壁树, 可食用, 弯曲, 热带, 营养, 热带水果, 蕉叶, 光滑..."
5,香蕉,黄色， 水果， 苹果， 猴子， 猩猩， 甜， 好吃， 剥皮， 通便， banana， 小黄人...,Simple,sukai/qwen_ppo_zh,好的，针对“香蕉”这个词，我的联想词列表如下：\n- 水果\n- 曲奇\n- 熟透\n- 香...,"水果, 曲奇, 熟透, 香焦, 黄色, 甜味, 印度, 椰子, 食用, 市场, 草莓"
...,...,...,...,...,...,...
4009,嗨,你好， 打招呼， 问好， 好， hello， hi， 开心， 哈喽， 微笑， 早上好， hi...,Simple,sukai/qwen_ppo_zh,好的，以下是“嗨”这个词的一些联想词：\n\n1. 你好\n2. 打招呼\n3. 感激\n4...,"你好, 打招呼, 感激, 朋友, 见面, 早上好, 晚上好, 热烈, 问候, 欢呼"
4011,嗨,你好， 打招呼， 问好， 好， hello， hi， 开心， 哈喽， 微笑， 早上好， hi...,Simple,sukai/llama_ppo_zh,"好玩! 我们来玩这个词联想实验吧。给出的词是 ""嗨""。\n\n以下是快速联想到的词:\n\n...","问候, 高兴, 话, 打招呼, 运动, handling (专业用语：计量器, scient..."
4013,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Simple,sukai/qwen_ppo_zh,好的，对于词语“悔改”，以下是一些可能的联想词：\n\n1. 罪恶\n2. 反思\n3. 天...,"罪恶, 反思, 天堂, 原谅, 新开始, 错误, 到悔悟, 坦白, 忏悔, 失误"
4014,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Complex,sukai/llama_ppo_zh,悔改 \n深感遗憾\n后悔\n忏悔\n反思\n检讨\n悔恨\n感谢赦免...,"悔改, 深感遗憾, 后悔, 忏悔, 反思, 检讨, 悔恨, 感谢赦免, 忏�πτωση, 主..."


In [13]:
with open('./swow_zh_results_processed.pkl', 'wb') as f:
    pickle.dump(swow_zh_results, f)